# Dependencies

In [123]:
import pickle
import numpy as np
import scipy.special
from glob import glob
import os
import re
import json


from sat.scripts.utils.misc import talk_to_me
from sat.scripts.utils.misc import make_output_dir

# Temp data

In [127]:
class args:
    pass


BASE = "/Users/jnom/Documents/Research/Doudna_lab/projects/MAIN_structure_project/software_dev/2024-11-24_pickle_to_contact_probability"
args.in_dir = f"{BASE}/6A6I_1__6A6I_2_colabfold_output_dir"
args.distance_cutoff = 8
args.outfile = f"{BASE}/6A6I_1__6A6I_2.tsv"

# Functions

In [136]:
def read_distogram(pkl_file):
    """
    Input: path to pickle file
    Output: Distogram
    """
    with open(pkl_file, "rb") as f:
        data = pickle.load(f)

    distogram = data.get("distogram", None)
    if distogram:
        return distogram
    else:
        msg = "Cannot detect a distogram within the pickle object. Something is wrong."
        raise ValueError(msg)

def sum_contact_probabilities(distogram, length_cutoff):
    """
    Summarize contact probabilities for residue pairs within a given distance cutoff.

    Parameters:
    - distogram (dict): AlphaFold distogram object containing "logits" and "bin_edges".
    - length_cutoff (float): Distance cutoff (e.g., 8.0 or 12.0 Å).

    Returns:
    - np.ndarray: A 2D array (N x N) of summed contact probabilities for residue pairs.
    
    More lay-man description. This function does the following:
    - The input is the distogram, which has an array of bin edges and a 3D matrix of
      logits. 

      The bin edges correspond to different distance cutoffs bins. There are D bins
      with a cutoff (e.g. 2A, 6A, 9A,...) 
      
      The logits matrix has shape N x N x D + 1, where N is equal to the total 
      number of residues in the input structure, while D is equal to the total number of
      distance bins (excluding the outlier distance bin). Basically, NxN array 1 
      corresponds to logits from the smallest distance bin, NxN array 2 is the next one,
      etc etc. The extra dimension of D is the logits (that could be converted to
      probability) of the residue-residue contact being above the final distance bin.

    - This function first uses the softmax function to convert logits to probability.
      Basically, each cell is exponentialized with exp() (solve for e^value). Then,
      each of those values is normalized across all of the D+1 bins - e.g. the first
      row/first col cell of the first bin is normalized to the first row/first col 
      cell of all other bins. Now, you still have a NxNxD matrix, but each value is the
      probability that the residue-residue contact is within that distance bin.

    - This function goes a final step - it sums all of the probabilities for each 
      residue-residue cell for all bins within the distance cutoff. So, you first find
      using the bin edges which bins are below distance cutoff, then sum those. This 
      in essence is finding the probability for each residue-residue cell that it 
      falls below that distance cutoff. So, the final output is an NxN matrix.
    """
    # Extract logits and bin_edges from distogram
    logits = distogram.get("logits")
    bin_edges = distogram.get("bin_edges")
    
    if logits is None or bin_edges is None:
        raise ValueError("Distogram must contain 'logits' and 'bin_edges'.")
    
    # Convert logits to probabilities using softmax
    probabilities = scipy.special.softmax(logits, axis=-1)  # Shape: (N, N, D+1)
    
    # Identify bins that correspond to distances ≤ length_cutoff
    contact_bins = np.where(bin_edges <= length_cutoff)[0]
    
    if len(contact_bins) == 0:
        raise ValueError(f"No bins found for the given length cutoff: {length_cutoff}")
    
    # Sum probabilities across the selected bins
    summed_probabilities = np.sum(probabilities[:, :, contact_bins], axis=-1)  # Shape: (N, N)
    
    return summed_probabilities

def highest_contact_probability(summed_probabilities, len1):
    """
    Extract the submatrix representing contacts between Protein 1 and Protein 2 and return the highest contact probability.

    Parameters:
    - summed_probabilities (np.ndarray): A 2D array (N x N) of contact probabilities.
    - len1 (int): Length of Protein 1. Protein 2 is assumed to have length (N - len1).

    Returns:
    - float: The highest contact probability in the submatrix.

    More lay-man description. This function does the following:
    - previously, we converted logits to probabilities, then summed the probabilities 
      for bins within the indicated distance cutoff. So, the input summed_probabilities
      is an NxN matrix. Here, we are finding the submatrix m' that corresponds to 
      the residue-residue contacts between protein chains. This evaluates to
      m' = m[: len1][len1:len1 + len2]. This is bascially all of the residue-residue
      contacts between chains.
    - this function extracts that submatrix, and then finds the highest
      value in that submatrix. This is the highest residue-residue contact probability
      between the two chains.
    - See Figure S6 of 10.1126/science.abm4805 for a visual representation/source
      methods.
    """
    # Total length of the summed_probabilities matrix
    total_len = summed_probabilities.shape[0]
    
    # Ensure len1 is valid
    if len1 <= 0 or len1 >= total_len:
        raise ValueError("Invalid Protein 1 length (len1). Ensure 0 < len1 < total_len.")
    
    # Define the length of Protein 2
    len2 = total_len - len1

    # Extract the submatrix m'
    submatrix = summed_probabilities[:len1, len1:len1 + len2]

    # Find the highest contact probability in the submatrix
    max_contact_probability = np.max(submatrix)
    
    return max_contact_probability

def parse_pr1_len(a3m_file_path):
    """
    This function parses the colabfold a3m file to retreive protein 1 length. This
    is present as the first line of the colabfold a3m file, which looks like:
    #98,76  1,1
    Where this input file contained two proteins, one 98 residues and one 76 residues.

    This function would return 98 in this example.
    """
    
    with open(a3m_file_path) as infile:
      first_line = infile.readline().strip()
      pr1_len = int(first_line.split("\t")[0].lstrip("#").split(",")[0])
      return int(pr1_len)

def pickle_name_to_info(input):
    """
    Takes in the name of a pickle file, and parses the prefix, model type, rank, and model.
    For example
    input: 6A6I_1__6H4B_1_all_rank_001_alphafold2_ptm_model_4_seed_000
    output: A tuple of 6A6I_1__6H4B_1, alphafold2_ptm, 1, 4 
    """
    if "_all_rank" not in input:
        msg = (
            "Cannot find the substring '_all_rank' in the name of the pickle, "
            f"{input}. Something is wrong."
            )
        raise ValueError(msg)
    sample_name = input.split("_all_rank")[0]
    rank_match = re.search(r'all_rank_(\d+)', input)
    model_match = re.search(r'model_(\d+)', input)
    model_type_match = re.search(r'all_rank_\d+_(.*?)_model_\d+', input)

    if not rank_match or not model_match or not model_type_match:
        msg = "Cannot parse the strings all_rank_ and/or model_ from the pickle file name, "
        msg += f"{input}. Something is wrong."
        raise ValueError(msg)

    rank = rank_match.group(1).lstrip("0")
    model = model_match.group(1)
    model_type = model_type_match.group(1)

    return sample_name, model_type, rank, model

def get_iptm_from_json(json_path):
    """
    Takes in path to a colabfold json file, and takes the iptm out of there.

    iPTM should be present for both AF monomer and mulitmer.
    """
    if not os.path.exists(json_path):
        raise ValueError(f"Cannot detect json file: {json_path}")

    with open(json_path) as infile:
        data = json.load(infile)
        if "iptm" not in data:
            msg = f"Cannot find iptm in the input json, {json_path}. It is expected "
            msg += "from both AF2 monomer and multimer."
            raise ValueError(msg)
        return str(data["iptm"])
    

# Main

In [137]:
talk_to_me("Getting protein length from a3m")
a3m = glob(f"{args.in_dir}/*a3m")
if len(a3m) > 1:
    raise ValueError("There are more than one a3m files in the directory, weird.")
pr1_len = parse_pr1_len(a3m[0])

talk_to_me("Iterating over pickle files")
output = ""
for FILE in glob(f"{args.in_dir}/*pickle"):
    basename = os.path.basename(FILE).replace(".pickle", "")
    sample_name, model_type, rank, model = pickle_name_to_info(basename)
    
    # Parsring the distogram from the pickle
    distogram = read_distogram(FILE)
    contact_prbabilities = sum_contact_probabilities(distogram, args.distance_cutoff)
    prob = str(highest_contact_probability(contact_prbabilities, pr1_len))

    # Also parse the iPTM from the json file corresponding to each pickle
    json_path = f"{FILE.rstrip('.pickle')}.json".replace("all", "scores")
    iptm = get_iptm_from_json(json_path)

    out_line = "\t".join([sample_name, model_type, rank, model, iptm, prob]) + "\n"
    output += out_line

talk_to_me("Writing output")
make_output_dir(args.outfile)
with open(args.outfile, "w") as outfile:
    outfile.write(output)

ipykernel_launcher.py: Getting protein length from a3m
ipykernel_launcher.py: Iterating over pickle files


# TESTS

'001_alphafold2_ptm_'

In [57]:
import pytest

class TestSumContactProbabilities:
    """Test suite for the sum_contact_probabilities function."""

    @pytest.fixture
    def example_distogram(self):
        """Fixture to provide a sample distogram for testing."""
        return {
            "logits": np.random.rand(5, 5, 64),  # Example logits for a small test case
            "bin_edges": np.linspace(2.3, 22.8, 63)  # Example bin edges
        }

    def test_basic_functionality(self, example_distogram):
        """Test basic functionality with a valid length cutoff."""
        length_cutoff = 8.0
        result = sum_contact_probabilities(example_distogram, length_cutoff)
        
        assert result.shape == (5, 5)
        assert np.all(result >= 0), "Contact probabilities should be non-negative."

    def test_length_cutoff(self, example_distogram):
        """Ensure probabilities are summed correctly for bins ≤ length_cutoff."""
        length_cutoff = 10.0
        logits = example_distogram["logits"]
        probabilities = softmax(logits, axis=-1)

        # Manually compute expected results for bins ≤ length_cutoff
        valid_bins = np.where(example_distogram["bin_edges"] <= length_cutoff)[0]
        expected = np.sum(probabilities[:, :, valid_bins], axis=-1)
        
        result = sum_contact_probabilities(example_distogram, length_cutoff)
        np.testing.assert_array_almost_equal(result, expected, decimal=5)

    def test_empty_distogram(self):
        """Test behavior when distogram is missing required keys."""
        incomplete_distogram = {"logits": np.random.rand(5, 5, 64)}  # Missing "bin_edges"
        with pytest.raises(ValueError, match="Distogram must contain 'logits' and 'bin_edges'."):
            sum_contact_probabilities(incomplete_distogram, 8.0)

    def test_no_bins_within_cutoff(self, example_distogram):
        """Ensure the function raises an error if no bins satisfy length_cutoff."""
        length_cutoff = 1.0  # Smaller than the smallest bin edge
        with pytest.raises(ValueError, match="No bins found for the given length cutoff"):
            sum_contact_probabilities(example_distogram, length_cutoff)

    def test_output_shape(self, example_distogram):
        """Check that the output matrix has the correct shape."""
        length_cutoff = 12.0
        logits_shape = example_distogram["logits"].shape
        result = sum_contact_probabilities(example_distogram, length_cutoff)
        
        assert result.shape == logits_shape[:2], "Output shape should match (L, L)."

    @pytest.fixture
    def small_distogram(self):
        """Fixture providing a small, readable distogram."""
        return {
            "logits": np.array([
                [[1, 2, 0, 0], [0, 1, 0, 0]],
                [[2, 1, 0, 0], [1, 0, 0, 0]]
            ]),  # 2x2 logits with 4 bins
            "bin_edges": np.array([2.0, 5.0, 10.0, 15.0])  # 4 bins
        }

    def test_small_distogram_with_cutoff_5(self, small_distogram):
        """Test with a small distogram and cutoff of 5.0 Å."""
        length_cutoff = 5.0
        result = sum_contact_probabilities(small_distogram, length_cutoff)
        
        # Expected output
        # Softmax probabilities along last axis
        probabilities = scipy.special.softmax(small_distogram["logits"], axis=-1)
        # Include bins corresponding to distances ≤ 5.0
        expected = np.sum(probabilities[:, :, :2], axis=-1)  # Use first two bins

        np.testing.assert_array_almost_equal(result, expected, decimal=5)

    def test_small_distogram_with_cutoff_10(self, small_distogram):
        """Test with a small distogram and cutoff of 10.0 Å."""
        length_cutoff = 10.0
        result = sum_contact_probabilities(small_distogram, length_cutoff)
        
        # Expected output
        # Softmax probabilities along last axis
        probabilities = scipy.special.softmax(small_distogram["logits"], axis=-1)
        # Include bins corresponding to distances ≤ 10.0
        expected = np.sum(probabilities[:, :, :3], axis=-1)  # Use first three bins

        np.testing.assert_array_almost_equal(result, expected, decimal=5)

class TestHighestContactProbability:
    """Test suite for the highest_contact_probability function using small arrays."""

    def test_highest_contact_probability_basic(self):
        """Test with a simple 2x2 contact probability matrix."""
        # Small readable summed probabilities matrix
        summed_probabilities = np.array([
            [0.1, 0.2, 0.3, 0.4],
            [0.5, 0.6, 0.7, 0.8],
            [0.9, 1.0, 1.1, 1.2],
            [1.3, 1.4, 1.5, 1.6]
        ])
        len1 = 2  # Protein 1 length
        # Submatrix: summed_probabilities[:2, 2:4] = [[0.3, 0.4], [0.7, 0.8]]
        expected = 0.8  # Highest value in submatrix

        result = highest_contact_probability(summed_probabilities, len1)
        assert result == expected, f"Expected {expected}, but got {result}."

    def test_highest_contact_probability_single_row(self):
        """Test when Protein 1 consists of a single residue."""
        # Small readable summed probabilities matrix
        summed_probabilities = np.array([
            [0.2, 0.3, 0.4, 0.5],
            [0.6, 0.7, 0.8, 0.9],
            [1.0, 1.1, 1.2, 1.3],
            [1.4, 1.5, 1.6, 1.7]
        ])
        len1 = 1  # Protein 1 length
        # Submatrix: summed_probabilities[:1, 1:] = [[0.3, 0.4, 0.5]]
        expected = 0.5  # Highest value in submatrix

        result = highest_contact_probability(summed_probabilities, len1)
        assert result == expected, f"Expected {expected}, but got {result}."

    def test_highest_contact_probability_equal_values(self):
        """Test when all values in the submatrix are the same."""
        # Small readable summed probabilities matrix
        summed_probabilities = np.array([
            [0.1, 0.1, 0.1, 0.1],
            [0.1, 0.1, 0.1, 0.1],
            [0.1, 0.1, 0.1, 0.1],
            [0.1, 0.1, 0.1, 0.1]
        ])
        len1 = 2  # Protein 1 length
        # Submatrix: summed_probabilities[:2, 2:4] = [[0.1, 0.1], [0.1, 0.1]]
        expected = 0.1  # All values are equal

        result = highest_contact_probability(summed_probabilities, len1)
        assert result == expected, f"Expected {expected}, but got {result}."

    def test_highest_contact_probability_invalid_lengths(self):
        """Test when Protein 1 length is invalid."""
        # Small readable summed probabilities matrix
        summed_probabilities = np.array([
            [0.2, 0.3, 0.4],
            [0.5, 0.6, 0.7],
            [0.8, 0.9, 1.0]
        ])
        len1 = 4  # Invalid length (exceeds matrix dimensions)

        with pytest.raises(ValueError, match="Invalid Protein 1 length"):
            highest_contact_probability(summed_probabilities, len1)